# 📝 Step 4B: LLM-based QA Generation from Graph Edges

This notebook demonstrates generating Question-Answer pairs that **require multiple document regions** to answer.

## Pipeline:
1. Load processed graph JSON
2. Extract edges between regions
3. Build prompts from edge pairs
4. Generate QA via LLM
5. Self-verify and filter results

In [ ]:
# Setup paths and imports
import sys
import json
from pathlib import Path

# Add src to path
ROOT_DIR = Path.cwd().parent
SRC_DIR = ROOT_DIR / "src"
sys.path.insert(0, str(SRC_DIR))

# Import our modules
from qa.llm_qa_generator import (
    LLMQAPromptBuilder, 
    QAVerifier, 
    LLMQAGenerator,
    GeneratedQA,
    ReasoningType
)

print("✅ Imports successful")

In [ ]:
# Load a sample processed JSON
OUTPUT_DIR = ROOT_DIR / "output" / "full_pipeline"

# Pick a validation sample
sample_files = list((OUTPUT_DIR / "validation").glob("*.json"))[:5]
print(f"Found {len(sample_files)} sample files")

# Load first valid file
sample_data = None
for f in sample_files:
    try:
        with open(f, 'r', encoding='utf-8') as fp:
            sample_data = json.load(fp)
            print(f"✅ Loaded: {f.name}")
            break
    except:
        continue

if sample_data:
    print(f"\n📊 Document structure:")
    print(f"   - Document ID: {sample_data.get('document_id')}")
    print(f"   - Regions: {len(sample_data.get('regions', []))}")
    print(f"   - Graph nodes: {len(sample_data.get('graph', {}).get('nodes', []))}")
    print(f"   - Graph edges: {len(sample_data.get('graph', {}).get('edges', []))}")

In [ ]:
# Explore graph structure
if sample_data:
    graph = sample_data.get('graph', {})
    nodes = graph.get('nodes', [])
    edges = graph.get('edges', [])
    
    print("🔗 Edge types in document:")
    edge_types = {}
    for edge in edges:
        rel = edge.get('relation', 'unknown')
        edge_types[rel] = edge_types.get(rel, 0) + 1
    
    for rel, count in sorted(edge_types.items(), key=lambda x: -x[1]):
        print(f"   - {rel}: {count}")
    
    print(f"\n📦 Node types:")
    node_types = {}
    for node in nodes:
        nt = node.get('type', 'unknown')
        node_types[nt] = node_types.get(nt, 0) + 1
    
    for nt, count in sorted(node_types.items(), key=lambda x: -x[1]):
        print(f"   - {nt}: {count}")

In [ ]:
# Initialize QA Generator components
prompt_builder = LLMQAPromptBuilder()
verifier = QAVerifier()

print("📜 Available relation prompts:")
for rel in prompt_builder.RELATION_PROMPTS.keys():
    print(f"   - {rel}")

print(f"\n✅ Prompt builder initialized")

In [ ]:
# Build node lookup
if sample_data:
    node_lookup = {n['id']: n for n in nodes}
    
    # Find a good edge for demo
    demo_edge = None
    for edge in edges:
        source_id = edge.get('source')
        target_id = edge.get('target')
        
        source_node = node_lookup.get(source_id)
        target_node = node_lookup.get(target_id)
        
        if source_node and target_node:
            source_text = source_node.get('text', '')
            target_text = target_node.get('text', '')
            
            # Find edge with substantial text content
            if len(source_text) > 20 and len(target_text) > 20:
                demo_edge = edge
                break
    
    if demo_edge:
        print("🔗 Selected edge for demo:")
        print(f"   Relation: {demo_edge.get('relation')}")
        print(f"   Source ({source_node['id']}): {source_text[:100]}...")
        print(f"   Target ({target_node['id']}): {target_text[:100]}...")

In [ ]:
# Build a prompt for the selected edge
if demo_edge and source_node and target_node:
    prompt = prompt_builder.build_prompt(
        source_node={
            'node_id': source_node['id'],
            'region_type': source_node.get('type', 'TextBlock'),
            'text': source_text
        },
        target_node={
            'node_id': target_node['id'],
            'region_type': target_node.get('type', 'TextBlock'),
            'text': target_text
        },
        relation=demo_edge.get('relation', 'spatial')
    )
    
    print("📝 Generated Prompt:")
    print("=" * 60)
    print(prompt[:2000])  # Show first 2000 chars
    if len(prompt) > 2000:
        print(f"\n... [{len(prompt) - 2000} more characters]")

In [ ]:
# Example: Simulated LLM response (replace with actual LLM call)
# In production, use: generator.generate_qa_for_edge(edge)

simulated_llm_response = '''{
    "question": "What is the relationship between the two text regions?",
    "answer": "Based on the spatial positioning, these regions are adjacent and likely part of the same document section.",
    "evidence_region_ids": [1, 2],
    "evidence_quotes": ["First region text", "Second region text"],
    "reasoning_type": "coreference",
    "reasoning_explanation": "The regions share common terms and spatial proximity."
}'''

print("💡 Simulated LLM Response:")
print(simulated_llm_response)

In [ ]:
# Verify the response
import json as json_lib

try:
    response_dict = json_lib.loads(simulated_llm_response)
    
    # Create GeneratedQA object
    qa_result = GeneratedQA(
        question=response_dict['question'],
        answer=response_dict['answer'],
        evidence_region_ids=response_dict['evidence_region_ids'],
        evidence_quotes=response_dict['evidence_quotes'],
        reasoning_type=ReasoningType(response_dict['reasoning_type']),
        confidence=0.8,
        relation_used='spatial'
    )
    
    # Verify
    print("🔍 Verification Results:")
    
    # Check consistency
    is_consistent, score = verifier.consistency_check(
        qa_result, 
        source_text="Sample source text",
        target_text="Sample target text"
    )
    print(f"   Consistency: {'✅' if is_consistent else '❌'} (score: {score:.2f})")
    
    # Check reject rules
    reject_reason = verifier.check_reject_rules(qa_result)
    if reject_reason:
        print(f"   ❌ REJECTED: {reject_reason.value}")
    else:
        print(f"   ✅ PASSED all reject rules")
        
except Exception as e:
    print(f"❌ Error: {e}")

## 🚀 Production Usage Example

To use with a real LLM, configure the generator:

In [ ]:
# Production usage (requires LLM API key)
"""
# Example with OpenAI
import openai

def call_openai(prompt: str) -> str:
    response = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": LLMQAPromptBuilder.SYSTEM_PROMPT},
            {"role": "user", "content": prompt}
        ],
        temperature=0.7
    )
    return response.choices[0].message.content

# Create generator with LLM function
generator = LLMQAGenerator(
    llm_call_fn=call_openai,
    min_confidence=0.7
)

# Process a document
qa_pairs = generator.process_document(sample_data)
print(f"Generated {len(qa_pairs)} QA pairs")
"""
print("💡 See code cell for production LLM integration example")

In [ ]:
# Load and display prompt templates
templates_path = ROOT_DIR / "src" / "qa" / "llm_prompt_templates.json"

if templates_path.exists():
    with open(templates_path, 'r', encoding='utf-8') as f:
        templates = json.load(f)
    
    print("📋 Available Prompt Templates:")
    for name, template in templates['prompt_templates'].items():
        print(f"\n🔹 {template['id']}: {template['name']}")
        print(f"   Type: {template['reasoning_type']}")
        print(f"   Description: {template['description'][:80]}...")
else:
    print("❌ Templates file not found")

## 📊 Summary

This notebook provides the framework for:

1. **Prompt Building**: Convert graph edges to LLM prompts
2. **QA Generation**: Generate multi-region questions via LLM
3. **Self-Verification**: Validate answers against evidence
4. **Quality Filtering**: Reject hallucinations and trivial QAs

### 5 Prompt Templates:
| ID | Name | Type | Use Case |
|---|---|---|---|
| PT001 | Text ↔ Table | validation | Verify text claims against table data |
| PT002 | Figure ↔ Caption | coreference | Connect figures with descriptions |
| PT003 | Text ↔ Text | coreference | Resolve cross-paragraph references |
| PT004 | Table ↔ Table | comparison | Compare data across tables |
| PT005 | Form ↔ Conclusion | inference | Link form data to decisions |